# Write a program to construct a Bayesian network considering medical data. Use this model to demonstrate the diagnosis of heart patients using the standard Heart Disease Data Set.
**Computer Laboratory II (417526) – Information Retrieval – Practical 3**

Install once (Linux terminal):
```
pip install pgmpy scikit-learn pandas numpy
```

In [ ]:
import os, re, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
warnings.filterwarnings("ignore")

DATA_DIR = "data"
def get_file(name, url):
    # Return path to data/<name>; download it first if it is not there yet.
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print("Downloading", url)
        urllib.request.urlretrieve(url, path)
    return path

In [ ]:
# ---- DATASET (embedded in this notebook) ----
# The data file is stored inside this notebook (compressed), so no download or internet is needed.
# Running this cell writes it to  data/heart.csv
import base64, gzip, os
os.makedirs("data", exist_ok=True)
_DATA = """
H4sIANit+2oC/41aW44cuRH8X2BvUmgwk886zux4LAEWLGFHH3s3f/hIvoKLzMgHq1uGP6QZqdmsZD4iI4P1n3/9++3Lx/H58dfx/uP4+efH58
8/fnwe71+/fzv+/sfnMf/j4/3L8fPr27e396/Hx19v//xyfP/2tx8fb/84Pr99//FxvL+tj4+fb39++fj5+28tH3Rcf0o9OM/f00E1XX/zIx/z
Jx30+2+5Xz/5oHz9f8X/jn79zI+6VvFcVUg+matSkW93nn8/ysG6qra55iC+/p2bfKOPufoxwqou379W5Vpk1bI1Pdq2aj2zXH9OllVF9ioHqf
UVT7lW8Qm7al52yYnF+uJ2tQy7Mk6Xpo+unVj8cJ2KzlPWN17Pq2EVvHV5itrATusMm+3FbOd84oTLtgf7qnUaeL5X+PdcT4yrYMtc1dSntFaF
J7Yisabr30TiBzn15XnzQx1yjst6HsiIJr512xHz6SuC7et784T7TrIqF2TNyocUbGr+NIblVFb+tZBZnplcejgfwe+y1yl7RY9WWTUzi2HVKX
vlmfGaWYQYksZw+QV+z5oN65sxlwt7DHnL5WAVWTbwOseMU5eftk9CFaaVV7oPWeUsmzpZluSEbEf2ldv58Dxiec6qpj37YBXNGkVecX7KmCq/
XXYVQq1JZS6PEvbK5is6dRUjk5/QIV1ePlEVTX1F9jxC1VPvVrXJvJ5jpWbNvmvduHm9In58RRlVLDalR0UuqxeyrCKs4mp+x6rS4AWGVWnFQF
al8ESp1HpFBxb29NqqGZvM2KmG2OTo9XpZ1c4N1cZzbGY2tIQT0hOOkkUwp4GaZ6scMpQxj5YKjyagjGcponOdjm2vIR6058GLC4mAtSUHRF6r
MnJ0WsVAonFDhsqGx5lrRI8N+YrtxHmzPKzR7nX5RftKbbenZWDVtYbkM6Jqa9J2tgHkSEDs9Ozv1SmxTzlDh1DU0zyXSvbYBS81DhiE6lm5KH
idrRrW+isipDW2KjpFLG7ZspMrh978wuN5xpcCqm/VXgZsv6LA2GncvKmZQjODkZuj7muydqNhsZcumV6wgHH5Sav43DGj44yzqyl+5LSvqajH
NPfRztBe9Ha1B5E7YbsyiULw0HWuhK6gdWxsY3hGEqF/3s9eLZNoYJ/xzJSA0SnB07RqiRauxqo01qVZ+Rz/yG5Ye/EtdxtOwjP+p3Gu17g681
sxoIfMzcHyi2wxA8arJbYs4UBZQBY58VMxWXttZKlGW6pp4TIgnNTd0UlVnVRBYBOaPsWCU4fPdqApvAgKb6tg+Wr62pLLbpWB11U4qcrv457c
SJQ0n6YAXkBxnYhoo2an3v18UW6EMkFQnoKLgrh8nGkEuhnLBNG5fvKJltAHGrlbhDS5dmIkNgkQbIDTUZb1qC2CJcXTNfjgsoi14Qtl80wpGB
guSBJ68ap4i4MAlf9JSJMUVBnB8kjFYNEsggq/di1f3seOq/Q5daMy9GpUmM2CNS81biVQBoEQHoBV0FOHW2Qusfpv7amQ4wRMGrvQ0Am9tNuj
xT1brg1BPcTWMyAtoJRxkKy8pz3acpzwZaqBiNLjfEkLuaDh2UCVA4HOi0ZwUfDylhMpbV6fMPKall8KCGa2Fkcg0fVWKRyeB/gqEncSbzyG71
SRfxOjzvYaCNsIIxAFwkAYI3QkWZY3AWfJGoI/aYuxEotmZJVuGIboL9vL5vX7aJakEmq5tUwfBJUoGbWqdwQLQyyjAiV6i36hNltxeqnom3Mg
aTmObzZ42sgV6qWGERY1We9tswef19dNTPEyKxLoEB492UD7Z6aAVlj+Os41G1ikQ70YX0vzyJUcIxdyoBglzkoamK3G96F0ZvgvCHFRojNxdx
iFo1t7VjLITUeVfvPiuX670t5nzfOJmwnoTDaMtDsF4vuNVycxSB0Erh8SUhvBbCoK8eZGbzzCThsKLilYPgI09+IFACglJpRJ5Ho6FxmLteIl
00uSMl2bqnkH1eLzB591W7MnG0liZy2k/CJstIoop1+ASTnD1PQL5tVcc4E2cJzticA56xjGndJTXaMWk5tGhltkUpEQbxPDFo8RRCKjeVaPZw
09ZRMRrPqr5/+ge+cp7Hsp+dTBfUul2Z2a91RhI4/sGN+bYdLkC1IzDTjpyZSMSdjIW+5hUVYxq1sJ+InnGXq35BKWyVyC8efzoMpBKmp3HucD
nwyxBDE2PY+OM715GPqIYMsaP8WQyRo02bne+ZAKgj2BpyfF3I2jqT8vT0HcIh0h1QfAFGE7avmtoJR9ZY0LWS+Mg0qUmvMv4En5Lrvs21+xaz
m1WSQMY3H5ggxu3nVUiGh30TcI1hpfRC49zRcXs9FMESF2EyvIdgL3ekb5PIw3mjBnrKn8/2ta9zzRcTYNq/OVK75qZaVC6hJnVkwmT0sbd7TO
I1n/aGBXCVcJADFlHiLWPgowIwWumtYITci569MHgYEl+NxlFGW3OhutOnbZitR6MsHddxoevbHPIcbYfRabTJyVh+Z1QsEV8UPyEybt/KtmH8
W9VZIhrPbeKWnToTNU2OkSVTkDiUxS9Oe5uJWtvUShMzm6mmSK3y+bqj9PUX9iVG+ODbs/a7W5JleXN1RwQAR15luSfbXzJayQNeScJ6fY0N1y
9fmkD7ZmiZqPGvIlBek/mxyUkKE4XylORVS0qbHp36Knp8P1RwGjTX4tM+e/pnUjFcE3q1R2zLFhw59iFXkmdJX5RkCXhB6q7LBaT6Mwc6uv0i
5PZjKr8s12FcRnbeR7Baof5pWfCm/SeR4j1LyfUKU76Q05VA1Ya59zRDOMmLVcvAJLC6K3ypgd6GF7GRLVhR4UPMobEhleNUFQuYfbTK9OcXTM
h47ZhCRsKaPtyAvoKWWWUap2kbmK9uCs4kpx/IYSL1Yh3cdcVewbFASRALXzyaafT7e3h+wkT2RPB209jYLAFoJDehuWtsTyZLBi1hOu1rQ1gH
IaebHE0pZvNp1+F6s8v/HNJjT3abe2m5qsKZOBhwc5+VCPhMGqrOcbfgdZ0iZWBSAqk3wiysUgLzQbRLnoKUrwlAKtAn7Wws7Nbj88emhboM2n
ayx6PO+TOSdDXMI1Zd6TOFfvpiyBCamnYnzsk1U5F9vz1NFFXbB2LCF41bsyIXgrhSuuYd2mLCxvIAVxsV3cUarNTZVNVa2lJRUoPwGpKFzVri
lkO113pCIb/irYSQpO9w6PcKcn1MvhalGvH/NT+ZnSJLYH9WRLKr9ctOuEhcbKKSQ2SPak2oBzYu+B3S4IeJzWebZeqrFZTO60dBZ48axy7Geo
EmdDg0gamu4tMOvdzTqAEBiQwii7nz6KUbiljC1pOgEo0snktr3VzJ1yEKYpgnrvYadicgyZUBjD17xp1RYkTj0dJn+dokka2xnQ+rS7JtbRSE
mUNaPiLZLY6t4UhI1S8cLz+e0x7vCqNTnrW9lnkgymG48F5+jqgLa9kuB1NUWLkhw8CaqVurwZm2RVsFivpzJMR389eb3iIVdd9iIP9vErWmfz
7Qb5MVGGKVu0dFlbpc6d3FolI/m/5YOyN76JLade1zXT5G7zStLU1Av7BB9wrOLpg+7X+jtvrjZVZPWmWmfnC6yrukQjfuSdn5ayEC8dqpcm0x
LD6WrQbqtyT0u7PEyXNKa0cDZHXFFWOtVwxnTTMBGbVex8OKcXtz1hEruUFCmpq1lV5PetdSw58bTR1/kW38E8xVEzeVNQ2WIaFCqKoBWqSTUI
0yrddBBKc5OykXlTq8qZ0E800M2qeTGoAzCtfv3gYHvzBqpvOyXFup3CKrie9uZUujE8CGNTVVQFtdTApZ5G7iytViXIbTJClutGUv0P8gotQc
nrOochH8yh7NSGeJuQDaT1aVPf0ZvBhJdLNuJGmIqUw5Mpb94SpBdeUied4UUgvYEJ5G7mumYy10DuJMRsBMtumPBijibWjQbrKzfVyV0yLGBp
HOqdLC+YcUDg5GDOUaNMt/CqctH9At9uvbZUWTciFPB3G1EUodarFmV7lWu4r9TrtYTXhcqtij0LTCMoeKnToldtbmaNbNY71hvpnjdxqtLaHc
1t0LmmfblDZlDusuY97S5+A81cLbd8viQrA1wLEG+jwHie61n1A53Z+WF3oM6o16zaw4uN5ZkFcwnqY9vbvo4ws/13+ewMMifcbVKYKfs5XNVl
ay5407KV8J7IPpsMnzVOvWxdn0Y5yS6b9eqQlpR+Jxo9vAXStrdprKj+C/AIEqhALAAA
"""
with open("data/heart.csv", "wb") as f:
    f.write(gzip.decompress(base64.b64decode(_DATA.replace("\n", ""))))
print("data/heart.csv written:", os.path.getsize("data/heart.csv"), "bytes")

## Practical 3 – Bayesian network on medical data (Heart Disease)
A **Bayesian network** is a directed acyclic graph where each node is a random variable and each edge a conditional dependency. Each node has a **conditional probability distribution (CPD)**. Here:

```
age, sex, exang, cp  ──►  heartdisease  ──►  restecg, chol
```
We learn the CPDs from data by **maximum likelihood**, then use **Variable Elimination** to answer queries such as *P(heartdisease | evidence)*.

Continuous columns (`age`, `chol`) are first **discretised** into bins because the tables are discrete.
The notebook accepts either file: `heart.csv` (the common 14-column version) or the original `processed.cleveland.data`.

In [ ]:
# pgmpy renamed BayesianNetwork -> DiscreteBayesianNetwork in version 1.0
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesNet
except ImportError:
    from pgmpy.models import BayesianNetwork as BayesNet
from pgmpy.inference import VariableElimination
# NOTE: model.fit(data) uses Maximum Likelihood Estimation by default in every pgmpy version
# (the estimator class was renamed MaximumLikelihoodEstimator -> DiscreteMLE in pgmpy 1.x,
#  which is why we do not pass it explicitly).
def fit_mle(m, data):
    m.fit(data)
from sklearn.model_selection import train_test_split

COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
        "exang", "oldpeak", "slope", "ca", "thal", "num"]

def load_heart():
    if os.path.exists(os.path.join(DATA_DIR, "processed.cleveland.data")):
        # original UCI file: no header, '?' = missing, 'num' = 0 (healthy) .. 4
        df = pd.read_csv(os.path.join(DATA_DIR, "processed.cleveland.data"), header=None, names=COLS, na_values="?")
        df["heartdisease"] = (df["num"] > 0).astype(int)
        df = df.drop(columns="num")
        src = "UCI processed.cleveland.data"
    else:
        path = get_file("heart.csv", "https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv")
        df = pd.read_csv(path, encoding="utf-8-sig", na_values="?")
        # In this widely used re-coded version, target = 1 means NO disease (it is inverted
        # w.r.t. the UCI 'num' column), so flip it to get heartdisease = 1 -> disease present.
        df["heartdisease"] = 1 - df["target"]
        df = df.drop(columns="target")
        src = "heart.csv"
    return df, src

heart, src = load_heart()
print("Loaded", src, "->", heart.shape)
print("\nSample instances from the dataset:")
print(heart.head())
print("\nAttributes and datatypes")
print(heart.dtypes)
print("\nMissing values:", int(heart.isna().sum().sum()))
print("heartdisease counts:", heart["heartdisease"].value_counts().to_dict())

In [ ]:
# Clean + discretise
df = heart.dropna().copy()
df["age"]  = pd.cut(df["age"],  bins=[0, 40, 50, 60, 120], labels=[0, 1, 2, 3]).astype(int)      # <40, 40-49, 50-59, 60+
df["chol"] = pd.cut(df["chol"], bins=[0, 200, 240, 1000],  labels=[0, 1, 2]).astype(int)         # desirable / borderline / high
df = df[["age", "sex", "cp", "exang", "restecg", "chol", "heartdisease"]].astype(int)

# Model structure (same as the lab manual)
model = BayesNet([("age", "heartdisease"), ("sex", "heartdisease"), ("exang", "heartdisease"),
                  ("cp", "heartdisease"), ("heartdisease", "restecg"), ("heartdisease", "chol")])

print("Learning CPDs using Maximum Likelihood Estimators")
fit_mle(model, df)
print("Model is valid:", model.check_model())

print("\nCPD of heartdisease (first rows):")
print(str(model.get_cpds("heartdisease"))[:1200])

In [ ]:
infer = VariableElimination(model)

print("1. P(heartdisease | restecg = 1)")
print(infer.query(variables=["heartdisease"], evidence={"restecg": 1}))

print("\n2. P(heartdisease | cp = value) for every chest-pain code in the data")
for v in sorted(df["cp"].unique()):
    p = infer.query(variables=["heartdisease"], evidence={"cp": int(v)}, show_progress=False)
    print(f"   cp={v}: P(disease)={p.values[1]:.3f}")

print("\n3. P(heartdisease | exang = 1, sex = 1, age group = 3 (60+))")
print(infer.query(variables=["heartdisease"], evidence={"exang": 1, "sex": 1, "age": 3}))

In [ ]:
# Diagnosis on unseen patients: train on 80 %, predict 20 %
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["heartdisease"])
m2 = BayesNet(model.edges())
fit_mle(m2, train)
inf2 = VariableElimination(m2)
feats = [c for c in test.columns if c != "heartdisease"]
known = {c: set(m2.get_cpds(c).state_names[c]) for c in feats}   # states seen during training
preds = []
for _, row in test.iterrows():
    ev = {c: int(row[c]) for c in feats if int(row[c]) in known[c]}
    q = inf2.query(["heartdisease"], evidence=ev, show_progress=False)
    preds.append(q.state_names["heartdisease"][int(np.argmax(q.values))])
acc = (np.array(preds) == test["heartdisease"].values).mean()
print(f"Diagnosis accuracy on {len(test)} unseen patients: {acc:.2%}")